# 初创公司 A 轮融资成功率预测

**EDP V2.0 应用示例 · 一级市场研究场景**

本 notebook 演示如何用 EDP 框架，对一家初创公司在未来 12 个月内能否完成下一轮（A 轮）融资进行**概率态势感知**。

---

## ⚠️ 用途边界声明

本示例为**学术研究与教育用途**，演示概率推断方法本身。

- 输出的融资成功率是**统计研究估计**，不是投资建议；
- **不构成**对任何具体项目尽调、投资、估值或 LP 出资的决策建议；
- 早期投资存在本金全部损失风险，历史融资模式不保证未来结果；
- 实际投资决策须依赖专业尽调，并咨询持牌专业人士。

使用者须自行承担一切决策风险。

---

## 场景设定

**研究对象**：一家 B2B SaaS 初创公司，已完成天使轮，预测其在未来 12 个月内能否完成 A 轮融资。

**可能结果（近似全域）**：
1. `success` —— 成功完成 A 轮
2. `bridge` —— 仅完成过桥/延期（介于成功与失败之间）
3. `fail` —— 未能融资（转型/收缩/关闭）

**信息来源（多源证据）**：
- 团队信号（创始人背景、核心团队完整度）
- 市场信号（赛道 TAM 增长、政策环境）
- Traction 信号（MRR 增长率、净留存、客户数）
- 行业景气度（同期一级市场融资活跃度）
- 类比公司（同赛道近期融资节奏）

## 1. 环境与框架加载

In [ ]:
import os, sys, importlib.util

# 加载 EDP 包（仓库内 src/python）
_SRC = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src', 'python'))
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)

_spec = importlib.util.spec_from_file_location(
    'edp', os.path.join(_SRC, '__init__.py'), submodule_search_locations=[_SRC]
)
_edp = importlib.util.module_from_spec(_spec)
sys.modules['edp'] = _edp
_spec.loader.exec_module(_edp)

from edp import (
    EDP, GenericDomain, Outcome, Evidence,
    ProbabilityEngine, ConformalEngine, ConformalConfig,
)
print('EDP 版本:', _edp.__version__)

## 2. 定义问题域

三个结果构成近似全域（成功 / 过桥 / 失败），用有序链表示其递进关系。

In [ ]:
domain = GenericDomain([
    Outcome('fail',   '未完成融资'),
    Outcome('bridge', '仅过桥/延期'),
    Outcome('success','成功完成 A 轮'),
])
# 构建有序链：fail ↔ bridge ↔ success
domain.graph_type = 'chain'

# 先验（无信息先验：均匀）
prior = domain.get_prior()
print('先验概率:', prior)

## 3. 构造多源证据

每条证据指向一个具体结果（定向证据），并附来源可靠性（reliability）与自报信心（confidence）。

> 注：以下数值为**演示用合成数据**，不代表任何真实项目。

In [ ]:
evidence = [
    # 团队信号：连续创业者、核心团队完整 → 倾向 success
    Evidence('team_bg', 'expert',
             {'probability': 0.72}, outcome_id='success',
             confidence=0.75, reliability=0.7),

    # Traction：MRR 同比 +120%、净留存 115% → 倾向 success
    Evidence('mrr_growth', 'model',
             {'probability': 0.78}, outcome_id='success',
             confidence=0.85, reliability=0.8),

    # 客户集中度：头部客户占比偏高 → 存在风险，倾向 bridge
    Evidence('cust_concentration', 'model',
             {'probability': 0.55}, outcome_id='bridge',
             confidence=0.6, reliability=0.7),

    # 行业景气度：同期一级市场偏冷 → 倾向 bridge/fail
    Evidence('market_chill', 'api',
             {'probability': 0.58}, outcome_id='bridge',
             confidence=0.7, reliability=0.75),

    # 类比公司：同赛道近期 2 家完成 A 轮、1 家收缩 → mixed
    Evidence('peer_comp', 'expert',
             {'probability': 0.60}, outcome_id='success',
             confidence=0.55, reliability=0.6),
]
print(f'共 {len(evidence)} 条证据，涉及来源类型:',
      sorted({e.source_type for e in evidence}))

## 4. 一键分析：L0 → L7 完整流程

In [ ]:
edp = EDP(domain, {
    'conformal': {'method': 'aci', 'alpha': 0.1},  # L7 保形：目标 90% 覆盖
    'allocation': {'kelly_fraction': 0.1},          # 保守
})

result = edp.analyze(evidence=evidence, budget=0)  # budget=0：本场景不涉及资金分配

print('摘要:', result['summary'])
print()
print('融合后概率:')
for oid, p in sorted(result['probabilities'].items(), key=lambda x: -x[1]):
    print(f'  {oid:8s}: {p:.1%}')

## 5. 态势评估：共识、稳定性、模型多样性

In [ ]:
a = result['assessment']
print(f"共识度:      {a.consensus_score:.3f}")
print(f"稳定性分类:  {a.stability.value}")
print(f"融合方法:    {a.fusion_method}")
print(f"置信度:      {a.confidence:.3f}")
if a.anomaly_flags:
    print(f"异常来源:    {a.anomaly_flags}")
else:
    print('异常来源:    无')

# 模型多样性（DTVW）：判断证据是否冗余
from edp import DomainAwarenessEngine, EvidenceSource, EvidenceType, SourceReliability
from datetime import datetime
sources = [
    EvidenceSource(e.id, EvidenceType(e.source_type),
                   SourceReliability.C, datetime.now(),
                   {'probability': e.extract_probability()}, e.confidence)
    for e in evidence
]
div = DomainAwarenessEngine.model_diversity(sources)
print()
print('模型多样性分析:')
print(f"  多样性:        {div['diversity']:.3f}")
print(f"  冗余度:        {div['redundancy']:.3f}")
print(f"  有效来源数:    {div['effective_sources']:.2f} / {len(sources)}")

## 6. 保形预测集（L7：有限样本覆盖率保证）

保形预测输出一个**预测集**，在有限样本下以 ≥ 90% 的概率包含真实结果。
这是 2025 年概率预测前沿方法，对分布漂移鲁棒（ACI 模式）。

In [ ]:
pset = result['prediction_set']
print(f"预测集:        {pset.prediction_set}")
print(f"目标覆盖率:    {pset.coverage_target:.0%}")
print(f"方法:          {pset.method}")
print()
if pset.is_full:
    print('→ 预测集包含全部结果：当前证据不足以窄化判断，建议补充关键证据。')
elif len(pset.prediction_set) == 1:
    print('→ 预测集仅含一个结果：证据较强地指向该结果（仍非确定）。')
else:
    print('→ 预测集已收窄：在 90% 覆盖保证下排除了部分结果。')

## 7. 在线校准：随历史融资结果迭代

随着真实融资结果陆续揭晓，可用 `conformal_update` 在线更新保形层，
使未来预测集的覆盖率在分布漂移下仍收敛到目标。

In [ ]:
# 模拟：对历史若干项目的预测概率与实际结果，在线更新保形层
import random
random.seed(11)
historical = [
    ({'fail':0.2,'bridge':0.3,'success':0.5}, 'success'),
    ({'fail':0.4,'bridge':0.4,'success':0.2}, 'bridge'),
    ({'fail':0.6,'bridge':0.3,'success':0.1}, 'fail'),
    ({'fail':0.2,'bridge':0.3,'success':0.5}, 'success'),
    ({'fail':0.3,'bridge':0.4,'success':0.3}, 'bridge'),
]
for preds, actual in historical:
    edp.initialize()
    edp.probabilities = preds
    res = edp.conformal_update(actual)
    print(f"预测={preds} 实际={actual:8s} 覆盖={res['covered']} "
          f"滚动覆盖率={res['coverage_rate']:.2f}")

stats = edp.conformal_engine.coverage_stats()
print(f"\n最终: 经验覆盖率={stats['empirical_coverage']:.2f} "
      f"目标={stats['target_coverage']:.2f} 校准样本={stats['calibration_size']}")

## 8. 结论与边界

**本示例展示了**：
- 用 EDP 把「融资成败预测」分解为「结果 + 多源信号」，统一融合；
- 定向证据（指向具体结果）避免概率被无差别拉平；
- L7 保形预测集给出有覆盖率保证的结论范围，而非单一确定性断言；
- 模型多样性指标揭示证据冗余度，提示是否需补充独立信号。

**再次声明**：本示例为学术研究演示，合成数据，**不构成任何投资建议**。
早期投资存在本金全部损失风险。实际决策须依赖专业尽调与持牌专业人士。